# Task 3: Partition Pruning Experiment (Queries 1, 6)

- February 2024 queries are run on a month-partitioned Gold table and an identical unpartitioned Delta copy.

## Connect to Spark

In [9]:
import sys

from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import (
    DEFAULT_INTEGRATED_TABLE_PATH,
    INTEGRATED_TABLE_VIEW,
    execute_registered_query,
    register_integrated_taxi_trips,
)

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-PartitionPruning-Query1-6')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark

## Define the two layouts

- February is used because the data covers January-March 2024. 
- Partition prunnig by skipping January and March.

In [10]:
PARTITIONED_TABLE_PATH = DEFAULT_INTEGRATED_TABLE_PATH
UNPARTITIONED_TABLE_PATH = (
    's3a://delta-data/gold/gold_integrated_taxi_trips_unpartitioned'
)
QUERY_NAMES = (
    'query_01_monthly_zone_demand_february_2024',
    'query_06_monthly_taxi_demand_trend_february_2024',
)
REPETITIONS = 9

## Create the unpartitioned layout

In [11]:
partitioned_table = spark.read.format('delta').load(PARTITIONED_TABLE_PATH)

(
    partitioned_table.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(UNPARTITIONED_TABLE_PATH)
)

unpartitioned_table = (
    spark.read.format('delta').load(UNPARTITIONED_TABLE_PATH)
)

assert partitioned_table.count() == unpartitioned_table.count()
print('Unpartitioned layout created with the same number of rows.')

Unpartitioned layout created with the same number of rows.


## Measure both layouts

Each layout receives one warm-up followed by three timed runs.

In [12]:
def measure_layout(table_path):
    spark.catalog.clearCache()
    register_integrated_taxi_trips(spark, table_path)
    return {
        query_name: measure_registered_query(spark, query_name, REPETITIONS)
        for query_name in QUERY_NAMES
    }

partitioned_results = measure_layout(PARTITIONED_TABLE_PATH)
unpartitioned_results = measure_layout(UNPARTITIONED_TABLE_PATH)

for query_name in QUERY_NAMES:
    assert rows_match(
        partitioned_results[query_name]['rows'],
        unpartitioned_results[query_name]['rows'],
    ), f'Partitioned and unpartitioned results differ for {query_name}.'

print('Both layouts return the same results within float tolerance.')

Both layouts return the same results within float tolerance.


## Partitioned physical plans

In [13]:
register_integrated_taxi_trips(spark, PARTITIONED_TABLE_PATH)
for query_name in QUERY_NAMES:
    print(f'\n--- {query_name} ---')
    execute_registered_query(spark, query_name).explain('formatted')


--- query_01_monthly_zone_demand_february_2024 ---
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [pickup_date#14725, pickup_zone_name#14733, pickup_month#14727]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips]
PushedFilters: [IsNotNull(pickup_date), GreaterThanOrEqual(pickup_date,2024-02-01), LessThan(pickup_date,2024-03-01)]
ReadSchema: struct<pickup_date:string,pickup_zone_name:string>

(2) Filter
Input [3]: [pickup_date#14725, pickup_zone_name#14733, pickup_month#14727]
Condition : ((isnotnull(pickup_date#14725) AND (pickup_date#14725 >= 2024-02-01)) AND (pickup_date#14725 < 2024-03-01))

(3) Project
Output [2]: [substring(pickup_date#14725, 1, 7) AS pickup_month#14768, pickup_zone_name#147

## Unpartitioned physical plans

- It does not contain a `PartitionFilters` entry.

In [14]:
register_integrated_taxi_trips(spark, UNPARTITIONED_TABLE_PATH)
for query_name in QUERY_NAMES:
    print(f'\n--- {query_name} ---')
    execute_registered_query(spark, query_name).explain('formatted')


--- query_01_monthly_zone_demand_february_2024 ---
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [pickup_date#15047, pickup_zone_name#15055]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips_unpartitioned]
PushedFilters: [IsNotNull(pickup_date), GreaterThanOrEqual(pickup_date,2024-02-01), LessThan(pickup_date,2024-03-01)]
ReadSchema: struct<pickup_date:string,pickup_zone_name:string>

(2) Filter
Input [2]: [pickup_date#15047, pickup_zone_name#15055]
Condition : ((isnotnull(pickup_date#15047) AND (pickup_date#15047 >= 2024-02-01)) AND (pickup_date#15047 < 2024-03-01))

(3) Project
Output [2]: [substring(pickup_date#15047, 1, 7) AS pickup_month#15090, pickup_zone_name#15055]
Input [2]: [pickup_dat

## Benchmark summary

In [15]:
summary = []
for query_name in QUERY_NAMES:
    partitioned = partitioned_results[query_name]
    unpartitioned = unpartitioned_results[query_name]
    summary.append({
        'query': query_name,
        'partitioned_samples_seconds': [
            round(value, 3) for value in partitioned['samples_seconds']
        ],
        'partitioned_median_seconds': round(partitioned['median_seconds'], 3),
        'unpartitioned_samples_seconds': [
            round(value, 3) for value in unpartitioned['samples_seconds']
        ],
        'unpartitioned_median_seconds': round(unpartitioned['median_seconds'], 3),
        'partitioned_speedup_factor': round(
            unpartitioned['median_seconds'] / partitioned['median_seconds'],
            2,
        ),
    })
summary

[{'query': 'query_01_monthly_zone_demand_february_2024',
  'partitioned_samples_seconds': [1.267,
   0.981,
   1.194,
   1.005,
   1.003,
   1.009,
   0.972,
   0.835,
   0.935],
  'partitioned_median_seconds': 1.003,
  'unpartitioned_samples_seconds': [0.765,
   0.665,
   0.827,
   0.683,
   0.771,
   0.728,
   0.705,
   0.812,
   0.999],
  'unpartitioned_median_seconds': 0.765,
  'partitioned_speedup_factor': 0.76},
 {'query': 'query_06_monthly_taxi_demand_trend_february_2024',
  'partitioned_samples_seconds': [0.847,
   0.789,
   0.86,
   0.788,
   0.806,
   0.771,
   0.923,
   0.738,
   0.796],
  'partitioned_median_seconds': 0.796,
  'unpartitioned_samples_seconds': [0.695,
   0.686,
   0.786,
   0.657,
   0.703,
   0.713,
   0.632,
   0.624,
   0.698],
  'unpartitioned_median_seconds': 0.695,
  'partitioned_speedup_factor': 0.87}]

## Cleanup

In [16]:
spark.catalog.clearCache()
spark.stop()

26/09/20 14:03:58 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-dbb289fa-90f5-4917-b9ca-66aedffc4088/pyspark-c0d1395d-9354-4b64-8ab4-c4dc6b086998. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-dbb289fa-90f5-4917-b9ca-66aedffc4088/pyspark-c0d1395d-9354-4b64-8ab4-c4dc6b086998
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:191)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:116)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:94)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1048)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:70)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun